# VLM_Test — VLM pseudo-labels for the preference failure margin

End-to-end, cleaned-up pipeline: use **Qwen3-VL** pseudo-labels (± a few clean labels) to learn the
Bradley-Terry **failure margin** on the frozen transition-only world model, and study when/whether the
VLM labels help.

**Run from the repo root** with the **`vlm_qwen3`** kernel (`conda activate vlm_qwen3` — it has both the
DreamerV3 world-model stack and Qwen3-VL). Bulk label sets and cached latents live in `vlm_labeling/`.

Sections: 1) world model + encode pool · 2) VLM labeling (few-shot) · 3) BT margin · 4) clean-vs-VLM
ablation · 5) margin heatmaps · 6) systematic-error diagnostics.

## 0. Setup

In [1]:
import os, sys, re, pickle, argparse
os.environ.setdefault("CUDA_VISIBLE_DEVICES","1"); os.environ.setdefault("HF_HUB_OFFLINE","1")
sys.path.insert(0, os.getcwd()); sys.path.insert(0, "eais_hw2/dreamerv3-torch")
import numpy as np, torch, torch.nn as nn, gym
from torch.nn.utils import spectral_norm
import ruamel.yaml as yaml, tools, models
import matplotlib.pyplot as plt
import generate_data_dubins_sidewalk as gen
from generate_pref_labels_dubins_sidewalk import weighted_margin_np, PREF_PROFILES
VLM_DIR="vlm_labeling"; dev=torch.device("cuda")

D=pickle.load(open("data/pref_labels_set_a_50k.pkl","rb"))
imgs=D["imgs"]; margin=D["margin"].astype(np.float32); priv=D["priv"]; Mn=len(margin)
R,SW=float(D["R"]),float(D["SW"]); W=PREF_PROFILES["set_a"]
cfg=yaml.YAML(typ="safe",pure=True).load(open("eais_hw2/configs.yaml").read())["defaults"]
config=argparse.Namespace(**{k:tools.args_type(v)(v) for k,v in cfg.items()})
config.num_actions=3; config.device=dev; config.use_margin_head=False

def is_obstacle(k): return np.hypot(priv[k,0],priv[k,1])<R
def is_sidewalk(k): return abs(priv[k,1])>SW and not is_obstacle(k)
def is_safe(k): return margin[k]>0
def region(k):
    if is_obstacle(k): return "OBS"
    if priv[k,1]>SW: return "USW"
    if priv[k,1]<-SW: return "LSW"
    return "USF" if priv[k,1]>=0 else "LSF"
# fixed pool split (train pool for labels, val pool for held-out eval)
rng=np.random.default_rng(0); val_mask=np.zeros(Mn,bool); val_mask[rng.choice(Mn,int(0.2*Mn),replace=False)]=True
train_idx=np.where(~val_mask)[0]; val_idx=np.where(val_mask)[0]
print("states:",Mn,"| R",R,"SW",SW)

states: 4000 | R 0.5 SW 1.1


## 1. World model + encode the state pool to latents
The margin head lives on the frozen WM latent (544-d). Latents for the 4000 pool states are cached.

In [2]:
sz=config.size[0]
obs_space=gym.spaces.Dict({"obs_state":gym.spaces.Box(-1,1,(2,),np.float32),"image":gym.spaces.Box(0,255,(sz,sz,3),np.uint8)})
wm=models.WorldModel(obs_space,gym.spaces.Discrete(3),0,config).to(dev)
_ck=torch.load("checkpoints/wm_transition_only.pt",map_location=dev)
_sd={k[len("_wm._orig_mod."):] if k.startswith("_wm._orig_mod.") else k[4:] if k.startswith("_wm.") else k:v
     for k,v in _ck["agent_state_dict"].items() if "_wm" in k}
wm.load_state_dict(_sd,strict=True); wm.eval(); wm.dynamics.sample=False

@torch.no_grad()
def encode(im_arr, ob_arr, chunk=256):
    out=[]
    for k in range(0,len(im_arr),chunk):
        im=im_arr[k:k+chunk].astype(np.float32); ob=ob_arr[k:k+chunk].astype(np.float32); B=len(im)
        acs=np.zeros((B,1,3),np.float32); acs[:,:,1]=1.0
        d={"image":im[:,None],"obs_state":ob[:,None],"action":acs,"is_first":np.ones((B,1),np.float32),"is_terminal":np.zeros((B,1),np.float32)}
        d=wm.preprocess(d); emb=wm.encoder(d); post,_=wm.dynamics.observe(emb,d["action"],d["is_first"])
        out.append(wm.dynamics.get_feat(post)[:,0])
    return torch.cat(out,0)

_fc=f"{VLM_DIR}/pool_feats.pt"
feats=torch.load(_fc,map_location=dev) if os.path.exists(_fc) else encode(D["imgs"],D["obs_state"])
if not os.path.exists(_fc): torch.save(feats,_fc)
print("pool latents:",tuple(feats.shape))

Encoder CNN shapes: {'image': (128, 128, 3)}
Encoder MLP shapes: {'obs_state': (2,)}


Decoder CNN shapes: {'image': (128, 128, 3)}
Decoder MLP shapes: {'obs_state': (2,)}


Optimizer model_opt has 27134213 variables.
pool latents: (4000, 544)


/home/alina/projects/latent_pref_reachability/eais_hw2/dreamerv3-torch/tools.py:891: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


## 2. VLM labeling (few-shot recipe)

The best recipe from the prompt study: a **safety-guidance** paragraph + **4 in-context demo pairs**,
each with a rich reason and answer, then the query pair, queried in a randomized A/B order.
Bulk label sets were generated with `vlm_labeling/gen_*.py` and saved to `vlm_labeling/vlm_labels_5k.pkl`
(5000 pairs @ ~0.70 accuracy). The cell below loads Qwen3-VL and labels a few pairs as a live demo.

In [3]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
_MODEL="Qwen/Qwen3-VL-4B-Instruct"
proc=AutoProcessor.from_pretrained(_MODEL)
vlm=Qwen3VLForConditionalGeneration.from_pretrained(_MODEL,dtype=torch.bfloat16).to("cuda").eval()

GUIDE=("Safety guidance for this map: The car must avoid the red center obstacle and the two gray "
    "sidewalk strips. The red obstacle is more dangerous than the sidewalks. Among safe regions, "
    "being farther away from both the center obstacle and the sidewalks is safer, and the upper "
    "safe area is preferred over the lower safe area.")
ANS='End with a line exactly "Answer: A" or "Answer: B".'
def _pil(k): from PIL import Image; return Image.fromarray(imgs[k])
def _desc(k):
    if is_obstacle(k): return "overlapping the red center obstacle (the most dangerous location)"
    if is_sidewalk(k): return "on a gray sidewalk strip (a hazard, but less dangerous than the obstacle)"
    return f"in the safe region ({'upper' if priv[k,1]>0 else 'lower'} area)"
def _why(a,b): return (f"State A is {_desc(a)}; State B is {_desc(b)}. Ranking: overlapping the obstacle is "
        f"worst, then a sidewalk; among safe spots, farther from hazards and the upper area are preferred")
def _demo_block(seed=1):
    r=np.random.default_rng(seed); sf=[k for k in train_idx if is_safe(k)]; fl=[k for k in train_idx if not is_safe(k)]
    ob=[k for k in fl if is_obstacle(k)]; sd=[k for k in fl if is_sidewalk(k)]; ss=sorted(sf,key=lambda k:margin[k])
    dm=[(r.choice(sf),r.choice(ob)),(r.choice(ob),r.choice(sd)),(ss[-1],ss[len(ss)//2]),(r.choice(sd),r.choice(sf))]
    blk=[{"type":"text","text":GUIDE+"\n\nHere are solved examples:"}]
    for a,b in dm:
        ans="A" if margin[a]>margin[b] else "B"
        blk+=[{"type":"text","text":"Example - State A:"},{"type":"image","image":_pil(int(a))},
              {"type":"text","text":"State B:"},{"type":"image","image":_pil(int(b))},
              {"type":"text","text":f"Reason: {_why(int(a),int(b))}. Answer: {ans}\n"}]
    return blk
@torch.no_grad()
def _ask(content,mx=160):
    m=[{"role":"user","content":content}]; t=proc.apply_chat_template(m,tokenize=False,add_generation_prompt=True)
    vi,vv=process_vision_info(m); inp=proc(text=[t],images=vi,videos=vv,return_tensors="pt").to("cuda")
    n=int(inp["input_ids"].shape[1]); o=vlm.generate(**inp,max_new_tokens=mx,do_sample=False)
    return proc.tokenizer.decode(o[0][n:],skip_special_tokens=True)
def _parse(t):
    m=re.findall(r"[Aa]nswer\s*[:\-]?\s*\(?([AB])",t); return m[-1].upper() if m else None
def vlm_label(a,b,blk):
    "return 1 if VLM says state a is safer, else 0 (randomized presentation order)"
    if np.random.rand()<0.5:
        c=blk+[{"type":"text","text":"Now solve this. State A:"},{"type":"image","image":_pil(a)},{"type":"text","text":"State B:"},{"type":"image","image":_pil(b)},{"type":"text","text":ANS}]
        return 1 if _parse(_ask(c))=="A" else 0
    c=blk+[{"type":"text","text":"Now solve this. State A:"},{"type":"image","image":_pil(b)},{"type":"text","text":"State B:"},{"type":"image","image":_pil(a)},{"type":"text","text":ANS}]
    return 0 if _parse(_ask(c))=="A" else 1

# live demo: label 12 random train pairs
blk=_demo_block(); rr=np.random.default_rng(0); ok=0
for _ in range(12):
    a,b=int(rr.choice(train_idx)),int(rr.choice(train_idx))
    if a==b or abs(margin[a]-margin[b])<0.1: continue
    lab=vlm_label(a,b,blk); gt=int(margin[a]>margin[b]); ok+=(lab==gt)
print("labeled 12 demo pairs; matched GT on", ok)

/home/alina/miniconda3/envs/vlm_qwen3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 713/713 [00:00<00:00, 15493.41it/s]

labeled 12 demo pairs; matched GT on 8


## 3. Bradley-Terry failure margin (head + weighted loss)
544→512→512→1 MLP (SiLU + spectral-norm). Per-example weights let us mix clean and VLM labels. Eval = pairwise accuracy on a held-out ground-truth set + correlation with the true margin.

In [4]:
class MarginHead(nn.Module):
    def __init__(s,f,u=512,l=2):
        super().__init__(); d,m=f,[]
        for _ in range(l): m+=[spectral_norm(nn.Linear(d,u)),nn.SiLU()]; d=u
        s.body=nn.Sequential(*m); s.out=spectral_norm(nn.Linear(d,1))
    def forward(s,x): return s.out(s.body(x)).squeeze(-1)
def bt_loss(pred,target,w):
    lp=torch.log_softmax(pred,1); mu=torch.stack([target,1-target],1)
    return ((w*(-(mu*lp).sum(1))).sum())/(w.sum()+1e-8)
def sample_pairs(pool,n,min_gap,seed):
    r=np.random.default_rng(seed); P=[]
    while len(P)<n:
        i,j=int(r.choice(pool)),int(r.choice(pool))
        if i==j or abs(margin[i]-margin[j])<min_gap: continue
        P.append((i,j))
    return np.array(P)
_ep=sample_pairs(val_idx,3000,0.15,999)
_ei=torch.tensor(_ep[:,0]); _ej=torch.tensor(_ep[:,1]); _eg=torch.tensor((margin[_ep[:,0]]>margin[_ep[:,1]]).astype(np.float32))
@torch.no_grad()
def eval_margin(m):
    acc=(((m(feats[_ei.to(dev)])>m(feats[_ej.to(dev)])).float())==_eg.to(dev)).float().mean().item()
    r=m(feats[torch.tensor(val_idx,device=dev)]); gv=torch.tensor(margin[val_idx],device=dev)
    rz,gz=r-r.mean(),gv-gv.mean(); corr=(rz*gz).sum().item()/(rz.norm().item()*gz.norm().item()+1e-8)
    return acc,corr
def train_margin(I,J,Y,Wt,steps=3000,seed=0):
    I=torch.tensor(I,device=dev);J=torch.tensor(J,device=dev)
    Y=torch.tensor(Y,dtype=torch.float32,device=dev);Wt=torch.tensor(Wt,dtype=torch.float32,device=dev)
    torch.manual_seed(seed); m=MarginHead(feats.shape[1],config.units).to(dev); opt=torch.optim.Adam(m.parameters(),1e-3)
    N=len(I); bs=min(512,N)
    for _ in range(steps):
        b=torch.randint(N,(bs,),device=dev); pred=torch.stack([m(feats[I[b]]),m(feats[J[b]])],1)
        loss=bt_loss(pred,Y[b],Wt[b]); opt.zero_grad(); loss.backward(); opt.step()
    return m

## 4. Ablation: clean labels vs VLM labels vs mixes
Loads the precomputed 5000 VLM labels. Clean labels are oracle (from the true margin). `w` = per-example weight on VLM labels (clean weight = 1).

In [5]:
V=pickle.load(open(f"{VLM_DIR}/vlm_labels_5k.pkl","rb"))["rows"]
vi=np.array([r["i"] for r in V]); vj=np.array([r["j"] for r in V]); vlab=np.array([r["vlm_label"] for r in V],float)
print(f"VLM labels: N={len(V)}  acc_vs_GT={np.mean([r['vlm_label']==r['gt_label'] for r in V]):.3f}")
def run(n_clean,n_vlm,w_vlm,seed):
    r=np.random.default_rng(seed); I=[];J=[];Y=[];Wt=[]
    if n_vlm>0: s=r.choice(len(V),n_vlm,replace=False); I+=list(vi[s]);J+=list(vj[s]);Y+=list(vlab[s]);Wt+=[w_vlm]*n_vlm
    if n_clean>0:
        cp=sample_pairs(train_idx,n_clean,0.10,seed+7); I+=list(cp[:,0]);J+=list(cp[:,1])
        Y+=list((margin[cp[:,0]]>margin[cp[:,1]]).astype(float)); Wt+=[1.0]*n_clean
    return eval_margin(train_margin(I,J,Y,Wt,seed=seed))[0]
GRID=[("clean-only 100",100,0,0),("clean-only 1000",1000,0,0),
      ("vlm-only 1000",0,1000,1),("vlm-only 5000",0,5000,1),
      ("100 clean +1000 vlm w=1",100,1000,1),("100 clean +1000 vlm w=0.1",100,1000,0.1)]
print(f"\n{'config':30s} eval_acc")
for name,nc,nv,wv in GRID:
    a=np.mean([run(nc,nv,wv,s) for s in (0,1,2)]); print(f"{name:30s} {a:.3f}")
print("\n=> VLM-only plateaus ~0.74 regardless of count; clean dominates; naive mixing hurts.")

VLM labels: N=5000  acc_vs_GT=0.697

config                         eval_acc


clean-only 100                 0.924


clean-only 1000                0.986


vlm-only 1000                  0.734


vlm-only 5000                  0.756


100 clean +1000 vlm w=1        0.762


100 clean +1000 vlm w=0.1      0.793

=> VLM-only plateaus ~0.74 regardless of count; clean dominates; naive mixing hurts.


## 5. Learned-margin heatmaps (a few key configs)
Encodes a spatial (x,y) grid, applies the margin head, averages over headings. The full 6x5 (clean x VLM) matrix is `visualizations/vlm_margin_matrix.png`.

In [6]:
ng,nz=41,4; xs=np.linspace(-1.5,1.5,ng); ys=np.linspace(-1.5,1.5,ng); ths=np.linspace(0,2*np.pi,nz,endpoint=False)
emoji=gen.load_tree_emoji(); gimg=[];gob=[];gidx=[];GT=np.zeros((ng,ng))
for a0,x in enumerate(xs):
    for a1,y in enumerate(ys):
        GT[a0,a1]=weighted_margin_np(x,y,W,R,SW)
        for th in ths: gimg.append(gen.render_state(np.array([x,y,th],np.float32),1.0,0.05,emoji,sz)); gob.append([np.cos(th),np.sin(th)]); gidx.append((a0,a1))
gfeat=encode(np.array(gimg,np.uint8),np.array(gob,np.float32)); gidx=np.array(gidx)
def grid_margin(m):
    with torch.no_grad(): pf=m(gfeat).cpu().numpy()
    cube=np.zeros((ng,ng,nz))
    for k,(a0,a1) in enumerate(gidx): cube[a0,a1,k%nz]=pf[k]
    return cube.mean(2)
def ov(ax):
    ax.add_patch(plt.Circle((0,0),R,fill=False,color="k",lw=1.2)); ax.axhline(SW,color="k",ls="--",lw=.7);ax.axhline(-SW,color="k",ls="--",lw=.7)
    ax.set_aspect("equal");ax.set_xticks([]);ax.set_yticks([])
fig,ax=plt.subplots(1,4,figsize=(18,4.6)); ext=[-1.5,1.5,-1.5,1.5]
def _show(A,F,t):
    vm=max(abs(F).max(),1e-3); im=A.imshow(F.T,extent=ext,origin="lower",cmap="seismic",vmin=-vm,vmax=vm,interpolation="none"); A.set_title(t,fontsize=10); ov(A)
_show(ax[0],GT,"ground truth")
for A,(name,nc,nv,wv) in zip(ax[1:],[("clean-only 100",100,0,0),("vlm-only 2000",0,2000,1),("100clean+1000vlm w=0.1",100,1000,0.1)]):
    r=np.random.default_rng(0); I=[];J=[];Y=[];Wt=[]
    if nv>0: s=r.choice(len(V),nv,replace=False); I+=list(vi[s]);J+=list(vj[s]);Y+=list(vlab[s]);Wt+=[wv]*nv
    if nc>0:
        cp=sample_pairs(train_idx,nc,0.10,7); I+=list(cp[:,0]);J+=list(cp[:,1]); Y+=list((margin[cp[:,0]]>margin[cp[:,1]]).astype(float)); Wt+=[1.0]*nc
    m=train_margin(I,J,Y,Wt,seed=0); _show(A,grid_margin(m),f"{name}\nacc={eval_margin(m)[0]:.3f}")
plt.tight_layout(); plt.show()

## 6. Diagnostics: the systematic VLM error

The ~30% VLM label noise is **not random** — it's a strong spatial bias. Directional bias per region
(VLM-safer% − GT-safer%; `+` = VLM over-rates as safe).

In [7]:
i5=np.array([r["i"] for r in V]); j5=np.array([r["j"] for r in V])
vl5=np.array([r["vlm_label"] for r in V]); gt5=np.array([r["gt_label"] for r in V])
ri=np.array([region(k) for k in i5]); rj=np.array([region(k) for k in j5])
print(f"{'region':6} {'N':>5} {'VLM_safer%':>10} {'GT_safer%':>9} {'bias':>7}")
for a in ["OBS","LSW","USW","LSF","USF"]:
    mi=(ri==a)&(rj!=a); mj=(rj==a)&(ri!=a)
    vs=np.concatenate([vl5[mi],1-vl5[mj]]); gs=np.concatenate([gt5[mi],1-gt5[mj]])
    print(f"{a:6} {len(vs):5d} {vs.mean()*100:9.1f}% {gs.mean()*100:8.1f}% {(vs.mean()-gs.mean())*100:+6.1f}%")
print("\n=> upper sidewalk +51% (over-rated as safe), lower-safe -30% (under-rated).")
print("   Reasoning dumps (diagnose_vlm_reasoning.py): VLM calls an upper-sidewalk car 'safe upper area'")
print("   -> it does not perceive the TOP strip as a hazard. Prompt fixes barely help -> perception limit.")

region     N VLM_safer% GT_safer%    bias
OBS     2204       3.9%      9.0%   -5.1%
LSW     1448      45.1%     36.9%   +8.1%
USW     1423      87.6%     36.3%  +51.3%
LSF     1517      51.5%     82.1%  -30.5%
USF     1640      82.2%     98.8%  -16.6%

=> upper sidewalk +51% (over-rated as safe), lower-safe -30% (under-rated).
   Reasoning dumps (diagnose_vlm_reasoning.py): VLM calls an upper-sidewalk car 'safe upper area'
   -> it does not perceive the TOP strip as a hazard. Prompt fixes barely help -> perception limit.


In [8]:
# spatial over/under-rating map
NB=10; xe=np.linspace(-1.5,1.5,NB+1); ye=np.linspace(-1.5,1.5,NB+1); B=np.zeros((NB,NB)); C=np.zeros((NB,NB))
for k in range(len(V)):
    for st,vs,gs in [(i5[k],vl5[k],gt5[k]),(j5[k],1-vl5[k],1-gt5[k])]:
        bx=max(0,min(NB-1,np.searchsorted(xe,priv[st,0])-1)); by=max(0,min(NB-1,np.searchsorted(ye,priv[st,1])-1)); B[bx,by]+=vs-gs; C[bx,by]+=1
BM=B/np.maximum(C,1); vm=np.abs(BM).max()
fig,axx=plt.subplots(figsize=(5.5,5)); im=axx.imshow(BM.T,extent=[-1.5,1.5,-1.5,1.5],origin="lower",cmap="coolwarm",vmin=-vm,vmax=vm)
ov(axx); axx.set_title("VLM over/under-rating by location\n(red = called SAFER than truth)"); fig.colorbar(im,fraction=.046,pad=.04); plt.show()

## Summary

- **VLM-only margins plateau at a ~0.74 noise floor** no matter the label count (500→5000); clean labels
  dominate (100 clean → 0.92), and naive fusion only hurts, in proportion to the VLM:clean count ratio.
- The error is **systematic, not random**: the VLM **over-rates the upper sidewalk (+51%) and under-rates
  the lower-safe region (−30%)** — it fails to perceive the *top* gray strip as a hazard and over-applies
  an "upper = safe" prior. Prompting barely fixes it → a perception limit at 128 px.
- Implications: prompting won't fix it; would need higher-res / more salient sidewalk rendering / coords,
  or use the VLM only where it's reliable (obstacle, lower sidewalk) with manual labels elsewhere.

Bulk generation + full ablation/diagnostic scripts: `vlm_labeling/` (`gen_*.py`, `margin_*.py`,
`diagnose_*.py`). Figures: `visualizations/vlm_margin_matrix.png`, `vlm_margin_accgrid.png`,
`vlm_label_bias.png`.